In [ ]:
import triton
import triton.language as tl
import torch
import math

device = 'cuda' if  torch.cuda.is_available() else 'cpu'
one_dim_len = 1000000
x = torch.arange(one_dim_len, dtype=torch.float32).to(device)
y = torch.arange(one_dim_len, dtype=torch.float32).to(device)

@triton.jit
def vector_addition_kernel(
    a_p,
    b_p,
    c_p,
    num,
    BLOCK_SIZE : tl.constexpr
):
  pid = tl.program_id(0)
  block_id = pid*BLOCK_SIZE
  offsets = block_id + tl.arange(0, BLOCK_SIZE)
  mask = offsets<num

  a = tl.load(a_p + offsets, mask = mask)
  b = tl.load(b_p + offsets, mask = mask)

  c = a+b
  tl.store(c_p + offsets, c, mask=mask)

def vector_add(a : torch.Tensor, b: torch.Tensor) -> torch.Tensor:
  assert device == 'cuda'
  assert a.shape == b.shape
  assert a.is_cuda
  assert b.is_cuda
  assert a.is_contiguous()
  assert b.is_contiguous()
  num = a.numel()
  c = torch.empty_like(a)

  grid = lambda meta: (triton.cdiv(num, meta['BLOCK_SIZE']),)

  vector_addition_kernel[grid](
      a,
      b,
      c,
      num,
      BLOCK_SIZE = 256
  )
  return c

vector_add(x,y)

tensor([0.0000e+00, 2.0000e+00, 4.0000e+00,  ..., 2.0000e+06, 2.0000e+06,
        2.0000e+06], device='cuda:0')

In [ ]:
@triton.jit
def silu_kernel(
    x_p,
    out_p,
    num,
    BLOCK_SIZE : tl.constexpr
):
  pid = tl.program_id(0)
  block_id = pid*BLOCK_SIZE
  offsets = block_id + tl.arange(0, BLOCK_SIZE)
  mask = offsets<num

  x = tl.load(x_p + offsets, mask=mask)
  t = tl.sigmoid(x)
  x = x*t
  tl.store(out_p+offsets, x, mask=mask)

def silu(x : torch.Tensor):
  assert x.is_cuda
  assert x.is_contiguous()
  out = torch.empty_like(x)
  num = x.numel()

  grid = lambda meta : (triton.cdiv(num, meta['BLOCK_SIZE']),)

  silu_kernel[grid](
      x,
      out,
      num,
      BLOCK_SIZE = 1024
  )
  return out

silu(x)

tensor([0.0000e+00, 7.3106e-01, 1.7616e+00,  ..., 1.0000e+06, 1.0000e+06,
        1.0000e+06], device='cuda:0')

In [ ]:
@triton.jit
def gelu_kernel(
    x_p,
    out_p,
    num,
    BLOCK_SIZE : tl.constexpr
):
  pid = tl.program_id(0)
  block_id = pid*BLOCK_SIZE
  offsets = block_id + tl.arange(0, BLOCK_SIZE)
  mask = offsets<num

  xo = tl.load(x_p + offsets, mask=mask)
  x = xo.to(tl.float32)
  x2 = x*x*x*0.044715 + x
  x2 = x2*0.7978845608
  tanh_val = 2.0 * tl.sigmoid(2.0 * x2) - 1.0
  x2 = 0.5 * x * (1.0 + tanh_val)
  out = x2.to(xo.dtype)
  tl.store(out_p + offsets, x2, mask = mask)


def gelu(x : torch.Tensor):
  assert x.is_cuda
  assert x.is_contiguous()
  out = torch.empty_like(x)
  num = x.numel()

  grid = lambda meta : (triton.cdiv(num, meta['BLOCK_SIZE']),)

  gelu_kernel[grid](
      x,
      out,
      num,
      BLOCK_SIZE = 1024
  )
  return out

gelu(x)


tensor([0.0000e+00, 8.4119e-01, 1.9546e+00,  ..., 1.0000e+06, 1.0000e+06,
        1.0000e+06], device='cuda:0')

In [ ]:
@triton.jit
def matrix_add_kernel(
    x_ptr,
    y_ptr,
    out_ptr,
    M,
    N,
    stride_m, stride_n,
    BLOCK_M : tl.constexpr,
    BLOCK_N : tl.constexpr
):
  pid_m = tl.program_id(0)
  pid_n= tl.program_id(1)

  block_idm = pid_m*BLOCK_M
  block_idn = pid_n*BLOCK_N

  offsets_m = block_idm + tl.arange(0, BLOCK_M)
  offsets_n = block_idn + tl.arange(0, BLOCK_N)

  offset_matrix = offsets_m[:,None]*stride_m + offsets_n[None,:]*stride_n

  x_ptr = x_ptr + offset_matrix
  y_ptr = y_ptr + offset_matrix
  out_ptr = out_ptr + offset_matrix

  mask = (offsets_m[:,None] < M) & (offsets_n[None,:] <N)

  x = tl.load(x_ptr, mask=mask, other=0.0)
  y = tl.load(y_ptr, mask=mask, other=0.0)

  out = x+y

  tl.store(out_ptr, out, mask=mask)

def matrix_add(x : torch.Tensor, y : torch.Tensor):
  assert x.is_contiguous()
  assert y.is_contiguous()
  assert x.is_cuda
  assert y.is_cuda
  assert x.shape == y.shape

  M, N = x.shape
  out = torch.empty_like(x)

  BLOCK_M = 32
  BLOCK_N = 32

  grid = lambda meta : (triton.cdiv(M , meta['BLOCK_M']), triton.cdiv(N, meta['BLOCK_N']))

  matrix_add_kernel[grid](
      x,
      y,
      out,
      M,
      N,
      x.stride(0), x.stride(1),
      BLOCK_M=BLOCK_M,
      BLOCK_N=BLOCK_N,
  )
  return out

x2d = torch.arange(10000, dtype=torch.float32).view(100,100).to(device)
y2d = torch.arange(10000, dtype=torch.float32).view(100,100).to(device)
print(matrix_add(x2d,y2d))

tensor([[0.0000e+00, 2.0000e+00, 4.0000e+00,  ..., 1.9400e+02, 1.9600e+02,
         1.9800e+02],
        [2.0000e+02, 2.0200e+02, 2.0400e+02,  ..., 3.9400e+02, 3.9600e+02,
         3.9800e+02],
        [4.0000e+02, 4.0200e+02, 4.0400e+02,  ..., 5.9400e+02, 5.9600e+02,
         5.9800e+02],
        ...,
        [1.9400e+04, 1.9402e+04, 1.9404e+04,  ..., 1.9594e+04, 1.9596e+04,
         1.9598e+04],
        [1.9600e+04, 1.9602e+04, 1.9604e+04,  ..., 1.9794e+04, 1.9796e+04,
         1.9798e+04],
        [1.9800e+04, 1.9802e+04, 1.9804e+04,  ..., 1.9994e+04, 1.9996e+04,
         1.9998e+04]], device='cuda:0')


In [ ]:
@triton.jit
def oned_softmax_kernel(
    x_ptr,
    out_ptr,
    M, N,
    BLOCK_SIZE : tl.constexpr
):
  pid = tl.program_id(0)
  # assert that N is less than block size before using
  block_id = pid*N
  offsets = block_id + tl.arange(0, BLOCK_SIZE)
  mask = offsets < (pid+1)*N

  x = tl.load(x_ptr+offsets, mask=mask, other = -float('inf'))

  x_max = tl.max(x, axis = 0)
  x = x - x_max

  x_numerator = tl.exp(x)
  x_numerator_safe = tl.where(mask, x_numerator, 0.0)
  x_denominator = tl.sum(x_numerator_safe, axis = 0)

  out = x_numerator / x_denominator

  tl.store(out_ptr+offsets, out, mask=mask)

def oned_softmax(x: torch.Tensor, M, N):
  assert x.is_cuda
  assert x.is_contiguous()
  #print(x.shape)
  assert x.shape[0] == M*N
  assert N < 1024

  num = x.numel()
  out = torch.empty_like(x)

  grid = lambda meta: (M,)

  oned_softmax_kernel[grid](
      x,
      out,
      M, N,
      BLOCK_SIZE = 1024
  )

  return out

oned_softmax(x,1000,1000)


tensor([0.0000, 0.0000, 0.0000,  ..., 0.0855, 0.2325, 0.6321], device='cuda:0')

In [ ]:
@triton.jit
def matrix_transpose_kernel(
    x_ptr,
    out_ptr,
    M, N,
    BLOCK_M : tl.constexpr,
    BLOCK_N : tl.constexpr
):
  pid_m = tl.program_id(0)
  pid_n = tl.program_id(1)

  block_idm = pid_m*BLOCK_M
  block_idn = pid_n*BLOCK_N

  offsets_m = block_idm + tl.arange(0, BLOCK_M)
  offsets_n = block_idn + tl.arange(0, BLOCK_N)

  offset_matrix_x = offsets_m[:,None]*N + offsets_n[None,:]
  offset_matrix_out = offsets_n[:,None]*M + offsets_m[None,:]

  x_ptr = x_ptr + offset_matrix_x
  out_ptr = out_ptr + offset_matrix_out

  x_mask = (offsets_m[:,None]<M) &  (offsets_n[None,:]<N)
  out_mask = (offsets_n[:,None]<N) &  (offsets_m[None,:]<M)

  x = tl.load(x_ptr, mask=x_mask, other=0.0)

  out = tl.trans(x)
  tl.store(out_ptr, out, out_mask)

def matrix_transpose(x : torch.Tensor):
  assert x.is_contiguous()
  assert x.is_cuda

  out = torch.empty_like(x).T.contiguous()
  M, N = x.shape

  grid = lambda meta: (triton.cdiv(M, meta['BLOCK_M']),triton.cdiv(N, meta['BLOCK_N']))

  matrix_transpose_kernel[grid](
      x,
      out,
      M, N,
      BLOCK_M=32,
      BLOCK_N=32
  )

  return out

x2d = torch.arange(100000, dtype=torch.float32).view(1000,100).to(device)
print(x2d)
print(matrix_transpose(x2d))

tensor([[0.0000e+00, 1.0000e+00, 2.0000e+00,  ..., 9.7000e+01, 9.8000e+01,
         9.9000e+01],
        [1.0000e+02, 1.0100e+02, 1.0200e+02,  ..., 1.9700e+02, 1.9800e+02,
         1.9900e+02],
        [2.0000e+02, 2.0100e+02, 2.0200e+02,  ..., 2.9700e+02, 2.9800e+02,
         2.9900e+02],
        ...,
        [9.9700e+04, 9.9701e+04, 9.9702e+04,  ..., 9.9797e+04, 9.9798e+04,
         9.9799e+04],
        [9.9800e+04, 9.9801e+04, 9.9802e+04,  ..., 9.9897e+04, 9.9898e+04,
         9.9899e+04],
        [9.9900e+04, 9.9901e+04, 9.9902e+04,  ..., 9.9997e+04, 9.9998e+04,
         9.9999e+04]], device='cuda:0')
tensor([[0.0000e+00, 1.0000e+02, 2.0000e+02,  ..., 9.9700e+04, 9.9800e+04,
         9.9900e+04],
        [1.0000e+00, 1.0100e+02, 2.0100e+02,  ..., 9.9701e+04, 9.9801e+04,
         9.9901e+04],
        [2.0000e+00, 1.0200e+02, 2.0200e+02,  ..., 9.9702e+04, 9.9802e+04,
         9.9902e+04],
        ...,
        [9.7000e+01, 1.9700e+02, 2.9700e+02,  ..., 9.9797e+04, 9.9897e+04,
        

In [ ]:
@triton.jit
def tiled_matmul_kernel(
    x_ptr,
    y_ptr,
    out_ptr,
    M, K, N,
    stride_xm, stride_xk,
    stride_yk, stride_yn,
    stride_outm, stride_outn,
    BLOCK_M : tl.constexpr,
    BLOCK_N : tl.constexpr,
    BLOCK_K : tl.constexpr
):

  # set out important assetions before hand for all the shapes
  pid_m = tl.program_id(0)
  pid_n = tl.program_id(1)

  offsets_m = pid_m*BLOCK_M + tl.arange(0, BLOCK_M)
  offsets_n = pid_n*BLOCK_N + tl.arange(0, BLOCK_N)
  offsets_k = tl.arange(0, BLOCK_K)

  offset_matrix_x = offsets_m[:,None]*stride_xm + offsets_k[None,:]*stride_xk
  offset_matrix_y = offsets_k[:,None]*stride_yk + offsets_n[None,:]*stride_yn
  offset_matrix_out = offsets_m[:,None]*stride_outm + offsets_n[None,:]*stride_outn

  x_ptr = x_ptr + offset_matrix_x
  y_ptr = y_ptr + offset_matrix_y
  out_ptr = out_ptr + offset_matrix_out

  out = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

  for k in range(0, K, BLOCK_K):
    mask_x = (offsets_m[:,None] < M) & (offsets_k[None,:]<(K-k))
    mask_y = (offsets_k[:,None] < (K-k)) & (offsets_n[None,:]<N)

    x = tl.load(x_ptr, mask=mask_x, other=0.0)
    y = tl.load(y_ptr, mask=mask_y, other=0.0)

    out += tl.dot(x,y)

    x_ptr += BLOCK_K*stride_xk
    y_ptr += BLOCK_K*stride_yk

  out_mask = (offsets_m[:,None] < M) & (offsets_n[None,:]<N)
  tl.store(out_ptr, out, mask=out_mask)

def tiled_matmul(x : torch.Tensor, y : torch.Tensor):
  M,K = x.shape
  Kt, N = y.shape
  assert K==Kt
  assert x.is_contiguous()
  assert y.is_contiguous()
  assert x.is_cuda
  assert y.is_cuda

  out = torch.empty((M,N), dtype=torch.float32).to(device).contiguous()

  grid = lambda meta : (triton.cdiv(M,meta['BLOCK_M']),triton.cdiv(N,meta['BLOCK_N']))

  tiled_matmul_kernel[grid](
      x,
      y,
      out,
      M,K,N,
      x.stride(0), x.stride(1),
      y.stride(0), y.stride(1),
      out.stride(0), out.stride(1),
      BLOCK_M=64,
      BLOCK_N=64,
      BLOCK_K=32
  )
  return out

x2d = torch.arange(12, dtype=torch.float32).to(device).view(3,4)
y2d = torch.arange(20, dtype=torch.float32).to(device).view(4,5)

tiled_matmul(x2d, y2d)

In [ ]:
@triton.jit
def online_softmax_kernel(
    x_ptr,
    y_ptr,
    M, N,
    stride_xm, stride_xn,
    stride_ym, stride_yn,
    BLOCK_SIZE : tl.constexpr
):
  pid = tl.program_id(0)

  offsets_x = pid*stride_xm + tl.arange(0,BLOCK_SIZE)*stride_xn()
  prev_max = -float('inf')
  denominator = 0.0

  for n in range(0, N, BLOCK_SIZE):
    mask = tl.arange(0,BLOCK_SIZE) < (N - n)
    block_x = tl.load(x_ptr+offsets_x, mask=mask, other = -float('inf'))

    block_max = tl.max(block_x, axis = 0)
    current_max = tl.maximum(block_max, prev_max)

    scaling_factor = tl.exp(prev_max-current_max)

    exponents = tl.exp(block_x-current_max)

    denominator = denominator*scaling_factor + tl.sum(exponents, axis= 0)

    prev_max = current_max

    offsets_x = offsets_x + BLOCK_SIZE*stride_xn

  offsets_x = pid*stride_xm + tl.arange(0,BLOCK_SIZE)*stride_xn()
  offsets_y = pid*stride_ym + tl.arange(0,BLOCK_SIZE)*stride_yn()
  inv_denom = 1.0 / denominator
  for n in range(0, N, BLOCK_SIZE):
    mask = tl.arange(0,BLOCK_SIZE) < (N - n)
    block_x = tl.load(x_ptr+offsets_x, mask=mask)
    normals = tl.exp(block_x-prev_max)*inv_denom
    tl.store(y_ptr+offsets_y,normals,mask=mask)
    offsets_x = offsets_x +BLOCK_SIZE*stride_xn
    offsets_y = offsets_y + BLOCK_SIZE*stride_yn

def online_softmax(x : torch.Tensor):
  assert x.is_cuda
  assert x.is_contiguous()

  y = torch.empty_like(x)
  M, N = x.shape

  grid = lambda meta : (M,)

  online_softmax_kernel[grid](
      x,
      y,
      M,N,
      x.stride(0), x.stride(1),
      y.stride(0), y.stride(1),
      BLOCK_SIZE = 1024
  )
  return y

In [ ]:
@triton.jit
def flash_attention_kernel(
    Q_ptr, # M D
    K_ptr, # N D
    V_ptr, # N D
    O_ptr, # M D
    M, N,
    stride_qm, stride_qd, stride_qb, stride_qh,
    stride_kn, stride_kd, stride_kb, stride_kh,
    stride_vn, stride_vd, stride_vb, stride_vh,
    stride_om, stride_od, stride_ob, stride_oh,
    HEADS_PER_BATCH : tl.constexpr,
    BLOCK_M : tl.constexpr,
    BLOCK_N : tl.constexpr,
    WHOLE_D : tl.constexpr,
    div_factor
):
  # ill be adding many comments because flash attention is challenging
  pid = tl.program_id(0)
  pid_bh = tl.program_id(1)
  # so for each kernel we have 4 "scanners", atleast thats what i like to call them. 1 is stationarry along Q
  # it scans BLOCK_M * D (whole channels) and for each pid it moves down
  # each pid is also responiblefor calulcating BLOCK_M * D of O, so this is stationary
  # scanning the transpose of k, we have D * BLOCK_N , moves along N
  # scanning v we have BLOCK_N * D, also moving along n

  current_batch = pid_bh // HEADS_PER_BATCH
  current_head = pid_bh % HEADS_PER_BATCH

  Q_ptr = Q_ptr + current_batch * stride_qb + current_head * stride_qh
  K_ptr = K_ptr + current_batch * stride_kb + current_head * stride_kh
  V_ptr = V_ptr + current_batch * stride_vb + current_head * stride_vh
  O_ptr = O_ptr + current_batch * stride_ob + current_head * stride_oh

  offsets_m = BLOCK_M * pid + tl.arange(0 , BLOCK_M) # stationary, not change
  offsets_n = tl.arange(0 , BLOCK_N) # moves along n, so we start
  offsets_d = tl.arange(0, WHOLE_D) # covers the enitre channels dimension

  mask_q = offsets_m < M # only have to do this once

  offsets_matrix_Q = offsets_m[:,None] * stride_qm + offsets_d[None,:] * stride_qd
  q_block = tl.load(Q_ptr + offsets_matrix_Q, mask_q[:, None], other = 0.0) # this is our Q block for this pid

  offsets_matrix_K = offsets_n[None,:] * stride_kn + offsets_d[:,None] * stride_kd

  offsets_matrix_V = offsets_n[:,None] * stride_vn + offsets_d[None,:] * stride_vd

  prev_max_per_row = tl.zeros([BLOCK_M], dtype=tl.float_32) - float('inf') # max for each row
  denominator_per_row = tl.zeros([BLOCK_M], dtype=tl.float_32)

  accumulator = tl.zeros([BLOCK_M, WHOLE_D], dtype=tl.float_32)

  inv_div_factor = 1/div_factor

  end_n = tl.minimum((pid+1)*BLOCK_M, N)

  for n in range(0, end_n, BLOCK_N):
    mask_kv = offsets_n < (N-n)
    causal_mask = (offsets_m[:, None]>=(n + offsets_n)[None, :]) & mask_kv[None, :]

    k_block = tl.load(K_ptr + offsets_matrix_K, mask_kv[None, :], other = 0.0 )
    v_block = tl.load(V_ptr + offsets_matrix_V, mask_kv[:, None], other = 0.0)

    qk_block = tl.dot(q_block,k_block)*inv_div_factor
    qk_block = tl.where(causal_mask, qk_block, -float('inf'))

    temp_max_per_row = tl.max(qk_block,axis=1)

    current_max_per_row = tl.maximum(temp_max_per_row, prev_max_per_row)

    score = tl.exp(qk_block-current_max_per_row[:, None])

    alpha_per_row = tl.exp(prev_max_per_row-current_max_per_row)

    denominator_per_row = denominator_per_row*alpha_per_row + tl.sum(score, axis = -1)

    p = tl.dot(score.to(v_block.dtype), v_block)

    accumulator = accumulator*alpha_per_row[:, None] + p

    prev_max_per_row = current_max_per_row

    offsets_matrix_K = offsets_matrix_K + BLOCK_N*stride_kn
    offsets_matrix_V = offsets_matrix_V + BLOCK_N*stride_vn

  accumulator = accumulator/ denominator_per_row[:, None]

  offsets_matrix_O = offsets_m[:, None]* stride_om + offsets_d[None, :] * stride_od
  tl.store(O_ptr + offsets_matrix_O, accumulator.to(q_block.dtype), mask_q[: , None])


def flash_attention(
    Q : torch.Tensor,
    K : torch.Tensor,
    V : torch.Tensor,
):
  assert Q.is_cuda and Q.is_contiguous()
  assert K.is_cuda and K.is_contiguous()
  assert V.is_cuda and V.is_contiguous()

  assert Q.shape[-1] == K.shape[-1]
  assert K.shape == V.shape

  B, H, T, C = Q.shape
  _, _, T2, _ = K.shape

  dk = math.sqrt(C)

  grid = lambda meta : (triton.cdiv(T, meta['BLOCK_M']),B*H)

  O = torch.empty_like(Q)

  flash_attention_kernel[grid](
      Q,
      K,
      V,
      O,
      T, T2,
      Q.stride(2), Q.stride(3), Q.stride(0), Q.stride(1),
      K.stride(2), K.stride(3), K.stride(0), K.stride(1),
      V.stride(2), V.stride(3), V.stride(0), V.stride(1),
      O.stride(2), O.stride(3), O.stride(0), O.stride(1),
      HEADS_PER_BATCH = H,
      BLOCK_M = 32,
      BLOCK_N = 32,
      WHOLE_D = C,
      dk

  )

  return O
